# AI Visual Novel — Exploratory Data Analysis (EDA)

Systematic exploration of three data sources:
1. **`game_design.json`** — Story graph structure, characters, scenes
2. **`story.txt`** — Script content, dialogue distribution, expressions
3. **`RAG knowledge base`** — Wikipedia corpus statistics

In [ ]:
import json
import re
import os
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

# ── Path setup ──────────────────────────────────────────────────────────────
ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path("..").resolve()
DESIGN_PATH   = ROOT / "data" / "game_design.json"
STORY_PATH    = ROOT / "data" / "story.txt"
RAG_DIR       = ROOT / "data" / "rag"
EXPR_PATH     = ROOT / "data" / "character_expressions.json"

# ── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
# Use a font that supports CJK characters if available
import matplotlib.font_manager as fm
cjk_fonts = [f.name for f in fm.fontManager.ttflist if "Han" in f.name or "Heiti" in f.name or "PingFang" in f.name]
if cjk_fonts:
    plt.rcParams["font.family"] = cjk_fonts[0]
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

PALETTE = sns.color_palette("Set2", 10)
print("✅ Environment ready")
print(f"   ROOT       : {ROOT}")
print(f"   design.json: {DESIGN_PATH.exists()}")
print(f"   story.txt  : {STORY_PATH.exists()}")
print(f"   RAG dir    : {RAG_DIR.exists()}")

---
## Part 1 — EDA on `game_design.json`

Analysing story graph topology, character settings and scene information.

In [ ]:
# ── Load game_design.json ────────────────────────────────────────────────────
with open(DESIGN_PATH, encoding="utf-8") as f:
    design = json.load(f)

nodes   = design["story_graph"]["nodes"]      # {node_id: node_dict}
edges   = design["story_graph"]["edges"]      # [{"from":..,"to":..,"choice_text":..}]
chars   = design["characters"]
scenes  = design["scenes"]

# ── Build NetworkX DiGraph ────────────────────────────────────────────────────
G = nx.DiGraph()
for nid, ndata in nodes.items():
    G.add_node(nid, node_type=ndata.get("type", "normal"))
for e in edges:
    G.add_edge(e["from"], e["to"], choice=e.get("choice_text") or "")

# ── Basic statistics ──────────────────────────────────────────────────────────
num_nodes    = G.number_of_nodes()
num_edges    = G.number_of_edges()
out_degrees  = dict(G.out_degree())
in_degrees   = dict(G.in_degree())
avg_out_deg  = np.mean(list(out_degrees.values()))

node_types   = Counter(ndata.get("type","normal") for ndata in nodes.values())
endings      = [n for n, d in nodes.items() if d.get("type") == "ending"]
merge_nodes  = [n for n, d in nodes.items() if d.get("type") == "merge"]
choice_nodes = [n for n in nodes if out_degrees.get(n, 0) > 1]

# All root→ending paths via DFS
def all_paths(G, source, sinks):
    results = []
    def dfs(node, path, visited):
        if node in sinks:
            results.append(list(path)); return
        for nxt in G.successors(node):
            if nxt not in visited:
                path.append(nxt); visited.add(nxt)
                dfs(nxt, path, visited)
                path.pop(); visited.discard(nxt)
    dfs(source, [source], {source})
    return results

root = "root"
sink_set = set(endings)
paths = all_paths(G, root, sink_set)
path_lengths = [len(p) for p in paths]

print("=" * 50)
print(f"  Title          : {design['title']}")
print(f"  Nodes          : {num_nodes}")
print(f"  Edges          : {num_edges}")
print(f"  Characters     : {len(chars)}")
print(f"  Scenes         : {len(scenes)}")
print(f"  Endings        : {len(endings)}  {endings}")
print(f"  Merge nodes    : {len(merge_nodes)}  {merge_nodes}")
print(f"  Choice nodes   : {len(choice_nodes)}  {choice_nodes}")
print(f"  Avg out-degree : {avg_out_deg:.2f}")
print(f"  Total paths    : {len(paths)}")
if path_lengths:
    print(f"  Path lengths   : min={min(path_lengths)}, max={max(path_lengths)}, mean={np.mean(path_lengths):.1f}")
else:
    print("  Path lengths   : N/A (no complete root→ending paths found)")
print(f"  Merge node %   : {len(merge_nodes)/num_nodes*100:.1f}%")
print("=" * 50)

In [ ]:
# ── Fig 1: DAG Structure Visualization ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

# Hierarchical layout using topological generations
try:
    pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
except Exception:
    # Fallback: manual layered layout
    topo = list(nx.topological_sort(G))
    layers = {}
    for n in topo:
        preds = list(G.predecessors(n))
        layers[n] = max((layers.get(p, 0) for p in preds), default=-1) + 1
    layer_members = defaultdict(list)
    for n, l in layers.items():
        layer_members[l].append(n)
    pos = {}
    for l, members in sorted(layer_members.items()):
        for i, n in enumerate(members):
            pos[n] = (l * 2, -(i - len(members) / 2))

# Color by node type
color_map = {"normal": "#74C0FC", "merge": "#FFD43B", "ending": "#69DB7C"}
node_colors = [color_map.get(nodes[n].get("type", "normal"), "#ADB5BD") for n in G.nodes()]
node_sizes  = [1800 if n in choice_nodes else 1200 for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, ax=ax, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight="bold", ax=ax)
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=18,
                       edge_color="#495057", width=1.5,
                       connectionstyle="arc3,rad=0.05")

# Edge labels (choice text, truncated)
edge_labels = {(e["from"], e["to"]): (e.get("choice_text") or "")[:8]
               for e in edges if e.get("choice_text")}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7,
                              font_color="#E03131", ax=ax)

# Legend
legend_handles = [mpatches.Patch(color=c, label=t) for t, c in color_map.items()]
legend_handles.append(mpatches.Patch(color="#74C0FC", label="choice node (larger)", linewidth=2))
ax.legend(handles=legend_handles, loc="upper right", fontsize=9)
ax.set_title(f"Story DAG — {design['title']}\n"
             f"{num_nodes} nodes | {num_edges} edges | {len(paths)} paths | {len(endings)} endings",
             fontsize=12, pad=12)
ax.axis("off")
plt.tight_layout()
plt.savefig(ROOT / "eval" / "fig1_dag.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eval/fig1_dag.png")

In [ ]:
# ── Fig 2: Node-type distribution + path-length histogram ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 2a: Node type pie
type_counts = Counter(ndata.get("type", "normal") for ndata in nodes.values())
axes[0].pie(type_counts.values(), labels=type_counts.keys(),
            colors=[color_map.get(t, "#ADB5BD") for t in type_counts.keys()],
            autopct="%1.1f%%", startangle=90, textprops={"fontsize": 10})
axes[0].set_title("Node Type Distribution")

# 2b: Out-degree bar
od_counter = Counter(out_degrees.values())
axes[1].bar(od_counter.keys(), od_counter.values(),
            color=PALETTE[:len(od_counter)], edgecolor="white")
axes[1].set_xlabel("Out-degree")
axes[1].set_ylabel("Number of nodes")
axes[1].set_title("Out-degree Distribution")
axes[1].set_xticks(sorted(od_counter.keys()))

# 2c: Path length histogram
if path_lengths:
    axes[2].hist(path_lengths, bins=range(min(path_lengths), max(path_lengths)+2),
                 color=PALETTE[2], edgecolor="white", align="left")
    axes[2].axvline(np.mean(path_lengths), color="crimson", linestyle="--",
                    label=f"mean={np.mean(path_lengths):.1f}")
    axes[2].legend(fontsize=9)
else:
    axes[2].text(0.5, 0.5, "No complete paths found", ha="center", va="center",
                 transform=axes[2].transAxes, fontsize=11, color="gray")
axes[2].set_xlabel("Path length (# nodes)")
axes[2].set_ylabel("Count")
axes[2].set_title(f"Path Length Distribution\n({len(paths)} total paths)")

plt.suptitle("game_design.json — Graph Statistics", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "eval" / "fig2_graph_stats.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eval/fig2_graph_stats.png")

In [ ]:
# ── Scene reuse analysis ──────────────────────────────────────────────────────
with open(STORY_PATH, encoding="utf-8") as f:
    story_raw = f.read()

# Count how many times each scene name appears in story.txt
scene_names = [s["name"] for s in scenes]
scene_usage = {name: len(re.findall(re.escape(name), story_raw)) for name in scene_names}

fig, ax = plt.subplots(figsize=(10, 4))
sorted_scenes = sorted(scene_usage.items(), key=lambda x: -x[1])
names_s, counts_s = zip(*sorted_scenes)
bars = ax.barh(names_s, counts_s, color=PALETTE[:len(names_s)])
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel("Times mentioned in story.txt")
ax.set_title("Scene Reuse Frequency")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(ROOT / "eval" / "fig3_scene_reuse.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"Scene reuse stats: max={max(counts_s)}, min={min(counts_s)}, mean={np.mean(counts_s):.1f}")
print("Saved: eval/fig3_scene_reuse.png")

---
## Part 2 — EDA on `story.txt`

Analysing dialogue distribution, per-character statistics, expressions and scene switches.

In [ ]:
# ── Parse story.txt ──────────────────────────────────────────────────────────
node_blocks = {}   # {node_id: text_block}
current_node = None
current_lines = []

for line in story_raw.split("\n"):
    m = re.match(r"=== Node: (\w+) ===", line)
    if m:
        if current_node:
            node_blocks[current_node] = "\n".join(current_lines)
        current_node = m.group(1)
        current_lines = []
    else:
        if current_node:
            current_lines.append(line)
if current_node:
    node_blocks[current_node] = "\n".join(current_lines)

# ── Per-node text length ──────────────────────────────────────────────────────
node_lengths = {nid: len(text) for nid, text in node_blocks.items()}

# ── Dialogue parsing ──────────────────────────────────────────────────────────
# <content id="CharName">text</content>
dialogue_pattern = re.compile(r'<content id="([^"]+)">(.*?)</content>', re.DOTALL)
all_dialogues = dialogue_pattern.findall(story_raw)    # [(speaker, text), ...]

speaker_lines  = Counter(sp for sp, _ in all_dialogues)
speaker_chars  = Counter()
for sp, txt in all_dialogues:
    speaker_chars[sp] += len(txt.strip())

# ── Expression tags ───────────────────────────────────────────────────────────
expr_pattern = re.compile(r'<image id="([^"]+)">(\w+)</image>')
all_exprs = expr_pattern.findall(story_raw)   # [(char, expression), ...]
expr_counter    = Counter(expr for _, expr in all_exprs)
char_expr_counter = Counter(f"{c}:{e}" for c, e in all_exprs)

# ── Scene tags ────────────────────────────────────────────────────────────────
scene_tags = re.findall(r'<scene>([^<]+)</scene>', story_raw)
scene_switches = Counter(scene_tags)

# ── Narration vs dialogue ─────────────────────────────────────────────────────
narration_lines = speaker_lines.get("旁白", 0)
dialogue_lines  = sum(v for k, v in speaker_lines.items() if k != "旁白")

print(f"Total nodes in story: {len(node_blocks)}")
print(f"Total dialogue lines : {sum(speaker_lines.values())}")
print(f"  Narration (旁白)   : {narration_lines}")
print(f"  Character dialogue : {dialogue_lines}")
print(f"  Narration ratio    : {narration_lines/(narration_lines+dialogue_lines)*100:.1f}%")
print(f"\nPer-speaker line counts:")
for sp, cnt in speaker_lines.most_common():
    chars_cnt = speaker_chars[sp]
    print(f"  {sp:20s}  lines={cnt:4d}  chars={chars_cnt:6d}")
print(f"\nExpression tag total : {len(all_exprs)}")
print(f"Expression types     : {dict(expr_counter.most_common(8))}")
print(f"\nScene switch total   : {len(scene_tags)}")
print(f"Unique scenes used   : {len(scene_switches)}")

In [ ]:
# ── Fig 4: Node text-length histogram ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

lengths_vals = list(node_lengths.values())
axes[0].bar(node_lengths.keys(), lengths_vals,
            color=[PALETTE[i % len(PALETTE)] for i in range(len(node_lengths))],
            edgecolor="white")
axes[0].set_xlabel("Node ID")
axes[0].set_ylabel("Text length (chars)")
axes[0].set_title("Per-node Script Length")
axes[0].tick_params(axis="x", rotation=45)
axes[0].axhline(np.mean(lengths_vals), color="crimson", linestyle="--",
                label=f"mean={np.mean(lengths_vals):.0f}")
axes[0].legend(fontsize=9)

# Box plot of lengths
axes[1].boxplot(lengths_vals, vert=True, patch_artist=True,
                boxprops=dict(facecolor=PALETTE[3], alpha=0.7))
axes[1].set_ylabel("Text length (chars)")
axes[1].set_title(f"Script Length Distribution\nmin={min(lengths_vals)} | max={max(lengths_vals)} | median={int(np.median(lengths_vals))}")
axes[1].set_xticks([])

plt.suptitle("story.txt — Node Script Length", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "eval" / "fig4_node_lengths.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eval/fig4_node_lengths.png")

In [ ]:
# ── Fig 5: Character dialogue bar chart + narration pie ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 5a: Line count bar
char_speakers = {k: v for k, v in speaker_lines.items() if k != "旁白" and k != "我"}
sorted_chars = sorted(char_speakers.items(), key=lambda x: -x[1])
names_c, lines_c = zip(*sorted_chars) if sorted_chars else ([], [])

axes[0].bar(names_c, lines_c,
            color=PALETTE[:len(names_c)], edgecolor="white")
axes[0].set_ylabel("Dialogue lines")
axes[0].set_title("Character Dialogue Line Count\n(excl. narration & protagonist)")
axes[0].tick_params(axis="x", rotation=20)
for i, v in enumerate(lines_c):
    axes[0].text(i, v + 0.5, str(v), ha="center", va="bottom", fontsize=9)

# 5b: Total character count bar (including protagonist "我")
sorted_chars_all = sorted(
    {k: v for k, v in speaker_chars.items() if k != "旁白"}.items(),
    key=lambda x: -x[1]
)
names_a, chars_a = zip(*sorted_chars_all) if sorted_chars_all else ([], [])
axes[1].bar(names_a, chars_a,
            color=PALETTE[:len(names_a)], edgecolor="white")
axes[1].set_ylabel("Total characters written")
axes[1].set_title("Character Total Text Volume\n(excl. narration)")
axes[1].tick_params(axis="x", rotation=20)

# 5c: Narration vs dialogue pie
pie_labels = ["Narration (旁白)", "Character Dialogue"]
pie_values = [narration_lines, dialogue_lines]
axes[2].pie(pie_values, labels=pie_labels, autopct="%1.1f%%",
            colors=[PALETTE[4], PALETTE[1]], startangle=90,
            textprops={"fontsize": 10})
axes[2].set_title("Narration vs Dialogue Ratio")

plt.suptitle("story.txt — Dialogue Analysis", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "eval" / "fig5_dialogue.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eval/fig5_dialogue.png")

In [ ]:
# ── Fig 6: Expression pie + heatmap (character × node) ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 6a: Expression type pie
top_exprs = expr_counter.most_common(8)
e_labels, e_vals = zip(*top_exprs) if top_exprs else ([], [])
axes[0].pie(e_vals, labels=e_labels, autopct="%1.1f%%",
            colors=PALETTE[:len(e_labels)], startangle=90,
            textprops={"fontsize": 9})
axes[0].set_title(f"Expression Tag Distribution\n(top {len(e_labels)}, total={sum(expr_counter.values())})")

# 6b: Character × node appearance heatmap
char_names_hm = [c["name"] for c in chars]
node_ids_hm   = list(node_blocks.keys())

# Count dialogue lines per (character, node)
matrix = np.zeros((len(char_names_hm), len(node_ids_hm)), dtype=int)
for nid, block in node_blocks.items():
    if nid not in node_ids_hm:
        continue
    j = node_ids_hm.index(nid)
    for sp, txt in dialogue_pattern.findall(block):
        # Map protagonist "我" → first protagonist character
        mapped_sp = sp
        if sp == "我":
            for c in chars:
                if c.get("is_protagonist"):
                    mapped_sp = c["name"]; break
        if mapped_sp in char_names_hm:
            i = char_names_hm.index(mapped_sp)
            matrix[i][j] += 1

sns.heatmap(matrix, ax=axes[1],
            xticklabels=node_ids_hm, yticklabels=char_names_hm,
            cmap="YlOrRd", annot=True, fmt="d", linewidths=0.4,
            cbar_kws={"label": "Dialogue lines"})
axes[1].set_title("Character × Node Dialogue Heatmap")
axes[1].set_xlabel("Story Node")
axes[1].set_ylabel("Character")
axes[1].tick_params(axis="x", rotation=45)

plt.suptitle("story.txt — Expression & Presence Analysis", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "eval" / "fig6_expr_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eval/fig6_expr_heatmap.png")

---
## Part 3 — EDA on RAG Knowledge Base

Analysing corpus composition: document counts, source types, character coverage.

In [ ]:
# ── Load all RAG JSON files ───────────────────────────────────────────────────
rag_files = list(RAG_DIR.glob("knowledge_*.json"))
all_docs   = []
ip_doc_counts = {}

for fp in rag_files:
    with open(fp, encoding="utf-8") as f:
        data = json.load(f)
    docs = data.get("documents", [])
    ip_name = fp.stem.replace("knowledge_", "").replace("_", " ")
    ip_doc_counts[ip_name] = len(docs)
    all_docs.extend(docs)

total_docs = len(all_docs)

# ── Type breakdown ─────────────────────────────────────────────────────────────
source_counter   = Counter(d["metadata"].get("source", "unknown") for d in all_docs)
type_counter     = Counter(d["metadata"].get("type",   "unknown") for d in all_docs)

# ── Per-character doc counts ──────────────────────────────────────────────────
char_doc_counts = Counter()
for d in all_docs:
    char = d["metadata"].get("character") or d["metadata"].get("title", "franchise")
    char_doc_counts[char] += 1

# ── Document text length stats ────────────────────────────────────────────────
doc_lengths = [len(d["text"]) for d in all_docs]

print(f"RAG files found    : {len(rag_files)}  {[f.name for f in rag_files]}")
print(f"Total documents    : {total_docs}")
print(f"\nSource breakdown   : {dict(source_counter)}")
print(f"Type breakdown     : {dict(type_counter)}")
print(f"\nPer-character/title doc counts:")
for k, v in char_doc_counts.most_common():
    print(f"  {k:35s}: {v}")
print(f"\nDoc text length    : min={min(doc_lengths)} | max={max(doc_lengths)} | mean={np.mean(doc_lengths):.0f}")

In [ ]:
# ── Fig 7: RAG corpus breakdown charts ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 7a: Docs per Wikipedia title (bar)
title_counts = Counter(
    d["metadata"].get("title", "unknown") for d in all_docs
)
sorted_titles = title_counts.most_common()
t_labels, t_vals = zip(*sorted_titles) if sorted_titles else ([], [])
bars7a = axes[0].barh(t_labels, t_vals,
                      color=PALETTE[:len(t_labels)], edgecolor="white")
axes[0].bar_label(bars7a, padding=3, fontsize=9)
axes[0].set_xlabel("Document fragments")
axes[0].set_title("Fragments per Wikipedia Page")
axes[0].invert_yaxis()

# 7b: character vs franchise type pie
type_labels = list(type_counter.keys())
type_vals   = list(type_counter.values())
axes[1].pie(type_vals, labels=type_labels, autopct="%1.1f%%",
            colors=PALETTE[:len(type_labels)], startangle=90,
            textprops={"fontsize": 10})
axes[1].set_title("Document Type Distribution\n(character vs franchise)")

# 7c: Doc text length histogram
axes[2].hist(doc_lengths, bins=15, color=PALETTE[5], edgecolor="white")
axes[2].axvline(np.mean(doc_lengths), color="crimson", linestyle="--",
                label=f"mean={np.mean(doc_lengths):.0f}")
axes[2].set_xlabel("Document text length (chars)")
axes[2].set_ylabel("Count")
axes[2].set_title("RAG Chunk Length Distribution")
axes[2].legend(fontsize=9)

plt.suptitle("RAG Knowledge Base — Corpus Statistics", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "eval" / "fig7_rag_corpus.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eval/fig7_rag_corpus.png")

In [ ]:
# ── Fig 8: Simulated BM25 retrieval coverage per character ────────────────────
# Simulate what keywords are used for retrieval and which characters' docs they hit
char_keywords = {c["name"]: [c["name"].lower(), c.get("role", "").lower()]
                 for c in chars}

retrieval_hits = {}  # {char_query: {source_char: count}}
for query_char in char_keywords:
    hits = Counter()
    query_terms = [t for t in char_keywords[query_char] if t.strip()]
    for d in all_docs:
        text_lower = d["text"].lower()
        score = sum(text_lower.count(term) for term in query_terms if term)
        if score > 0:
            source = d["metadata"].get("character") or d["metadata"].get("title", "?")
            hits[source] += 1
    retrieval_hits[query_char] = hits

# Build retrieval heatmap: rows = query character, cols = source character/title
query_chars  = list(retrieval_hits.keys())
source_chars = sorted(set(
    k for hits in retrieval_hits.values() for k in hits.keys()
), key=lambda x: -sum(retrieval_hits[q].get(x, 0) for q in query_chars))[:12]

ret_matrix = np.array([
    [retrieval_hits[q].get(s, 0) for s in source_chars]
    for q in query_chars
])

fig, ax = plt.subplots(figsize=(14, max(3, len(query_chars) * 0.8 + 2)))
im = sns.heatmap(ret_matrix, ax=ax,
                 xticklabels=source_chars, yticklabels=query_chars,
                 cmap="Blues", annot=True, fmt="d", linewidths=0.5,
                 cbar_kws={"label": "Matching doc fragments"})
ax.set_title("RAG Retrieval Coverage\n(row=query character, col=matched source)")
ax.set_xlabel("Source (Wikipedia title / character)")
ax.set_ylabel("Query character")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(ROOT / "eval" / "fig8_retrieval_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eval/fig8_retrieval_heatmap.png")

---
## Part 4 — Clustering Analysis

Two clustering tasks to answer whether the generated content is **truly diverse**:

1. **Node topic clustering** — do story branches actually explore different themes, or are they superficially distinct?
2. **Character dialogue style clustering** — do different characters truly "sound" different from each other?

In [ ]:
# ── Part 4-A: Node topic clustering ──────────────────────────────────────────
# Embed node summaries → K-Means → 2-D PCA scatter
# Goal: visually identify "pseudo-branches" (nodes that cluster together
#        despite being on nominally different story paths).

import sys, os
sys.path.insert(0, str(ROOT))

import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score

# ── 4-A.1: collect node summaries ────────────────────────────────────────────
sg_nodes = design.get("story_graph", {}).get("nodes", [])
node_ids  = [n["id"]      for n in sg_nodes if n.get("summary", "").strip()]
summaries = [n["summary"] for n in sg_nodes if n.get("summary", "").strip()]

print(f"Nodes with non-empty summary: {len(node_ids)}")

if len(node_ids) < 3:
    print("⚠️  Too few nodes with summaries for clustering. Skipping Part 4-A.")
else:
    # ── 4-A.2: embed summaries ────────────────────────────────────────────────
    # Try to use project's existing EmbeddingIndex; fall back to TF-IDF if unavailable.
    embeddings = None

    try:
        from agents.knowledge_builder import EmbeddingIndex
        npz_path = str(ROOT / "data" / "rag" / "_node_cluster_tmp.npz")
        ei = EmbeddingIndex(npz_path)
        if ei._google_client is not None or ei._openai_client is not None:
            vecs = []
            for summ in summaries:
                v = ei.embed_text(summ[:1000])
                vecs.append(v if v is not None else np.zeros(3072, dtype=np.float32))
            embeddings = np.stack(vecs).astype(np.float32)
            print(f"Embeddings via API: shape={embeddings.shape}")
        else:
            print("No embedding API key found; falling back to TF-IDF.")
    except Exception as e:
        print(f"Embedding unavailable ({e}); using TF-IDF.")

    if embeddings is None:
        # TF-IDF fallback
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(max_features=500, ngram_range=(1, 2), sublinear_tf=True)
        embeddings = vec.fit_transform(summaries).toarray().astype(np.float32)
        print(f"TF-IDF matrix: shape={embeddings.shape}")

    # ── 4-A.3: choose K via silhouette score ──────────────────────────────────
    X = normalize(embeddings)
    max_k = min(6, len(node_ids) - 1)
    sil_scores = {}
    for k in range(2, max_k + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        sil_scores[k] = silhouette_score(X, labels)

    best_k = max(sil_scores, key=sil_scores.get)
    print(f"\nSilhouette scores: { {k: f'{v:.3f}' for k, v in sil_scores.items()} }")
    print(f"Best K = {best_k}")

    # ── 4-A.4: final clustering + PCA projection ──────────────────────────────
    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    cluster_labels = km_final.fit_predict(X)

    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X)

    fig, ax = plt.subplots(figsize=(9, 7))
    palette = sns.color_palette("Set2", best_k)

    for k in range(best_k):
        mask = cluster_labels == k
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            color=palette[k], s=120, alpha=0.85,
            label=f"Cluster {k}  (n={mask.sum()})",
            edgecolors="white", linewidths=0.8,
        )
        # annotate node IDs
        for i, (x, y) in enumerate(coords[mask]):
            ax.annotate(node_ids[np.where(mask)[0][i]],
                        (x, y), fontsize=7, alpha=0.7,
                        xytext=(4, 4), textcoords="offset points")

    ax.set_title(
        f"Node Topic Clustering  (K={best_k}, PCA 2-D projection)\n"
        f"Variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}  "
        f"PC2={pca.explained_variance_ratio_[1]:.1%}",
        fontsize=12,
    )
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.legend(loc="best", fontsize=9)
    plt.tight_layout()
    plt.savefig(ROOT / "eval" / "fig9_node_clustering.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Saved: eval/fig9_node_clustering.png")

    # ── 4-A.5: pseudo-branch detection ───────────────────────────────────────
    from itertools import combinations

    edges_set = set()
    for e in design.get("story_graph", {}).get("edges", []):
        src = e.get("from", e.get("source", ""))
        dst = e.get("to",   e.get("target", ""))
        edges_set.add((src, dst))

    pseudo_branches = []
    for i, j in combinations(range(len(node_ids)), 2):
        if cluster_labels[i] == cluster_labels[j]:
            ni, nj = node_ids[i], node_ids[j]
            # Flag pairs on the same cluster that are NOT parent-child
            if (ni, nj) not in edges_set and (nj, ni) not in edges_set:
                sim = float(np.dot(X[i], X[j]))
                if sim > 0.80:
                    pseudo_branches.append((ni, nj, round(sim, 3)))

    if pseudo_branches:
        print(f"\n⚠️  {len(pseudo_branches)} potential pseudo-branch pair(s) "
              f"(same cluster, cosine sim > 0.80, no direct edge):")
        for a, b, s in sorted(pseudo_branches, key=lambda x: -x[2])[:10]:
            print(f"   {a}  ↔  {b}   sim={s}")
    else:
        print("\n✓ No pseudo-branch pairs detected — all clusters are well-separated.")

In [ ]:
# ── Part 4-B: Character dialogue style clustering ─────────────────────────────
# TF-IDF on each character's full dialogue → hierarchical clustering → dendrogram
# Goal: check whether different characters have genuinely distinct speech patterns.

import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# ── 4-B.1: extract per-character dialogue from story.txt ─────────────────────
_RE_CONTENT = re.compile(r'<content\s+id="([^"]+)">([^<]*)</content>', re.IGNORECASE)

char_corpus: dict = defaultdict(list)   # {display_name → [dialogue, ...]}
id_to_name  = {c.get("id", "").upper(): c.get("name", "") for c in chars}
id_to_name.update({c.get("name", "").upper(): c.get("name", "") for c in chars})

for m in _RE_CONTENT.finditer(story_text):
    cid  = m.group(1).strip().upper()
    text = m.group(2).strip()
    if cid in ("NARRATOR", "旁白") or not text:
        continue
    name = id_to_name.get(cid, m.group(1).strip())
    char_corpus[name].append(text)

# Filter to characters with at least 5 lines
valid_chars = {name: lines for name, lines in char_corpus.items() if len(lines) >= 5}
print(f"Characters with ≥5 lines: {list(valid_chars.keys())}")

if len(valid_chars) < 2:
    print("⚠️  Not enough characters with sufficient dialogue for clustering.")
else:
    char_names  = list(valid_chars.keys())
    char_docs   = [" ".join(valid_chars[n]) for n in char_names]

    # ── 4-B.2: TF-IDF features ───────────────────────────────────────────────
    # Use character-level n-grams (1-2 chars) to capture stylistic differences
    # beyond just topic: sentence length, punctuation density, repeated words.
    tfidf_word = TfidfVectorizer(
        max_features=300, ngram_range=(1, 2),
        sublinear_tf=True, min_df=1,
        analyzer="word",
    )
    tfidf_char = TfidfVectorizer(
        max_features=200, ngram_range=(2, 4),
        sublinear_tf=True, min_df=1,
        analyzer="char_wb",
    )

    W = tfidf_word.fit_transform(char_docs).toarray()
    C = tfidf_char.fit_transform(char_docs).toarray()
    # Concatenate word + char features
    X_style = np.hstack([W, C]).astype(np.float64)
    X_style = normalize(X_style)

    # ── 4-B.3: cosine similarity matrix ──────────────────────────────────────
    sim_matrix = cosine_similarity(X_style)

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(char_names) * 0.7 + 2)))

    # Heatmap
    ax = axes[0]
    sns.heatmap(
        sim_matrix, ax=ax,
        xticklabels=char_names, yticklabels=char_names,
        cmap="RdYlGn_r", vmin=0, vmax=1,
        annot=True, fmt=".2f", linewidths=0.5,
        cbar_kws={"label": "Cosine similarity"},
    )
    ax.set_title("Character Dialogue Style\nCosine Similarity Matrix", fontsize=11)
    ax.tick_params(axis="x", rotation=45)

    # ── 4-B.4: hierarchical clustering dendrogram ─────────────────────────────
    ax = axes[1]
    dist_matrix = 1 - np.clip(sim_matrix, 0, 1)
    np.fill_diagonal(dist_matrix, 0)
    condensed = squareform(dist_matrix, checks=False)
    Z = linkage(condensed, method="ward")

    dendrogram(
        Z,
        labels=char_names,
        orientation="left",
        color_threshold=0.5 * max(Z[:, 2]),
        ax=ax,
    )
    ax.set_title("Character Style Dendrogram\n(Ward linkage on TF-IDF)", fontsize=11)
    ax.set_xlabel("Distance")
    ax.axvline(x=0.5 * max(Z[:, 2]), color="grey", linestyle="--", alpha=0.6,
               label="Cluster threshold")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(ROOT / "eval" / "fig10_char_style_clustering.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Saved: eval/fig10_char_style_clustering.png")

    # ── 4-B.5: style distinctiveness summary ─────────────────────────────────
    print("\nStyle distinctiveness (avg pairwise cosine similarity — lower = more distinct):")
    off_diag = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
    print(f"  Mean similarity : {off_diag.mean():.3f}")
    print(f"  Max  similarity : {off_diag.max():.3f}")
    print(f"  Min  similarity : {off_diag.min():.3f}")

    if off_diag.max() > 0.80:
        pairs_similar = [
            (char_names[i], char_names[j], round(sim_matrix[i, j], 3))
            for i, j in zip(*np.triu_indices_from(sim_matrix, k=1))
            if sim_matrix[i, j] > 0.80
        ]
        print(f"\n⚠️  Highly similar character pairs (sim > 0.80):")
        for a, b, s in sorted(pairs_similar, key=lambda x: -x[2]):
            print(f"   {a}  ↔  {b}   sim={s}")
    else:
        print("\n✓ All character pairs have distinct dialogue styles (max sim ≤ 0.80).")

---
## Part 5 — Quality Evaluation (A/B Analysis)

Reads the logs written by `workflow.py` to answer **"does quality_scorer actually improve generation?"**

Three evidence sources:
1. **Score trajectory across Producer rounds** — do scores rise with each revision?
2. **Cross-run final score comparison** — are runs with scorer enabled consistently better?
3. **OOC score distribution** — which characters are most at risk of going out of character?

In [ ]:
# ── Part 5-A: Load quality logs ───────────────────────────────────────────────

import json
import pandas as pd
from pathlib import Path

QUALITY_DIR  = ROOT / "logs" / "quality"
ROUNDS_LOG   = QUALITY_DIR / "rounds.jsonl"
FINAL_LOG    = QUALITY_DIR / "final.jsonl"

SCORE_COLS = [
    "branch_balance_score",
    "character_balance_score",
    "scene_diversity_score",
    "overall_structural_score",
]
SCORE_LABELS = {
    "branch_balance_score":      "Branch Balance",
    "character_balance_score":   "Character Balance",
    "scene_diversity_score":     "Scene Diversity",
    "overall_structural_score":  "Overall",
}
SCORE_COLORS = {
    "branch_balance_score":      "#4C72B0",
    "character_balance_score":   "#DD8452",
    "scene_diversity_score":     "#55A868",
    "overall_structural_score":  "#C44E52",
}

def _load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rows = []
    for line in path.read_text(encoding="utf-8").strip().splitlines():
        line = line.strip()
        if line:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return pd.DataFrame(rows) if rows else pd.DataFrame()

rounds_df = _load_jsonl(ROUNDS_LOG)
final_df  = _load_jsonl(FINAL_LOG)

print(f"Rounds log  : {len(rounds_df)} records  ({'found' if not rounds_df.empty else 'EMPTY — run workflow first'})")
print(f"Final log   : {len(final_df)}  records  ({'found' if not final_df.empty else 'EMPTY — run workflow first'})")

if not rounds_df.empty:
    print(f"\nRuns in rounds log : {rounds_df['run_id'].nunique()}")
    print(rounds_df[["run_id", "game_title", "round", "producer_verdict", "overall_structural_score"]].to_string(index=False))
if not final_df.empty:
    print(f"\nRuns in final log  : {len(final_df)}")
    print(final_df[["run_id", "game_title", "scorer_enabled", "overall_structural_score"]].to_string(index=False))

In [ ]:
# ── Part 5-B: Score trajectory across Producer rounds ─────────────────────────
# Key question: do scores RISE across rounds when quality_scorer is enabled?
# Evidence: if slopes are positive, the quality feedback is effective.

if rounds_df.empty:
    print("⚠️  No rounds data yet. Run the workflow at least once.")
else:
    run_ids = rounds_df["run_id"].unique()
    n_runs  = len(run_ids)

    fig, axes = plt.subplots(
        1, len(SCORE_COLS), figsize=(5 * len(SCORE_COLS), 4),
        sharey=False,
    )
    if len(SCORE_COLS) == 1:
        axes = [axes]

    for ax, col in zip(axes, SCORE_COLS):
        label = SCORE_LABELS[col]
        color = SCORE_COLORS[col]

        for i, run_id in enumerate(run_ids):
            run_data = rounds_df[rounds_df["run_id"] == run_id].sort_values("round")
            if col not in run_data.columns or run_data[col].isna().all():
                continue

            scorer_on = run_data["scorer_enabled"].iloc[0] if "scorer_enabled" in run_data.columns else True
            linestyle = "-" if scorer_on else "--"
            alpha     = 0.9 if scorer_on else 0.5
            title_short = str(run_data["game_title"].iloc[0])[:20] if "game_title" in run_data else run_id

            ax.plot(
                run_data["round"], run_data[col],
                marker="o", linewidth=2, linestyle=linestyle, alpha=alpha,
                label=f"{title_short} ({'scorer' if scorer_on else 'baseline'})",
            )
            # Annotate PASS round
            pass_rows = run_data[run_data["producer_verdict"] == "PASS"]
            if not pass_rows.empty:
                pr = pass_rows.iloc[0]
                ax.axvline(pr["round"], color="grey", linestyle=":", alpha=0.5, linewidth=1)
                ax.annotate(
                    "PASS", xy=(pr["round"], pr[col]),
                    xytext=(4, 4), textcoords="offset points",
                    fontsize=7, color="grey",
                )

        ax.axhline(0.55, color="red", linestyle="--", linewidth=0.8, alpha=0.6, label="Threshold (0.55)")
        ax.set_title(label, fontsize=11)
        ax.set_xlabel("Producer Round")
        ax.set_ylabel("Score")
        ax.set_ylim(0, 1.05)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.legend(fontsize=7, loc="lower right")

    fig.suptitle(
        "Score Trajectory Across Producer Rounds\n"
        "(Solid = scorer enabled, Dashed = baseline; dashed red = quality threshold)",
        fontsize=12,
    )
    plt.tight_layout()
    plt.savefig(ROOT / "eval" / "fig11_score_trajectory.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Saved: eval/fig11_score_trajectory.png")

    # ── Quantitative summary: avg score delta per round ──────────────────────
    print("\nAvg score change per round (positive = improvement):")
    for col in SCORE_COLS:
        if col not in rounds_df.columns:
            continue
        deltas = []
        for run_id in run_ids:
            run_data = rounds_df[rounds_df["run_id"] == run_id].sort_values("round")[col].dropna().values
            if len(run_data) > 1:
                deltas.append(run_data[-1] - run_data[0])
        if deltas:
            print(f"  {SCORE_LABELS[col]:<22}: {sum(deltas)/len(deltas):+.4f}  (n={len(deltas)} runs)")

In [ ]:
# ── Part 5-C: Cross-run final score comparison ────────────────────────────────
# Key question: are scorer-enabled runs consistently higher quality?
# With enough runs this becomes a proper A/B comparison.

if final_df.empty:
    print("⚠️  No final quality data yet. Run the workflow at least once.")
else:
    avail_score_cols = [c for c in SCORE_COLS if c in final_df.columns]

    # ── Grouped bar chart ─────────────────────────────────────────────────────
    n_runs   = len(final_df)
    x        = np.arange(n_runs)
    bar_w    = 0.8 / len(avail_score_cols)

    fig, ax = plt.subplots(figsize=(max(8, n_runs * 1.8), 5))

    for i, col in enumerate(avail_score_cols):
        offset  = (i - len(avail_score_cols) / 2 + 0.5) * bar_w
        vals    = final_df[col].fillna(0).values
        # Color by scorer_enabled if column exists
        if "scorer_enabled" in final_df.columns:
            colors = ["#4C9F70" if s else "#AAAAAA" for s in final_df["scorer_enabled"]]
        else:
            colors = [SCORE_COLORS[col]] * n_runs

        bars = ax.bar(x + offset, vals, width=bar_w * 0.9,
                      color=colors if col == avail_score_cols[0] else SCORE_COLORS[col],
                      alpha=0.85, label=SCORE_LABELS[col], edgecolor="white", linewidth=0.5)
        # Value labels
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=7)

    ax.axhline(0.55, color="red", linestyle="--", linewidth=0.8, alpha=0.6, label="Threshold (0.55)")
    ax.set_xticks(x)
    run_labels = []
    for _, row in final_df.iterrows():
        title  = str(row.get("game_title", ""))[:15]
        run_id = str(row.get("run_id", ""))[-6:]
        sc     = " ✓" if row.get("scorer_enabled") else " ✗"
        run_labels.append(f"{title}\n[{run_id}]{sc}")
    ax.set_xticklabels(run_labels, fontsize=8)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Score (0–1)")
    ax.set_title(
        "Final Quality Scores per Generation Run\n"
        "(✓ = scorer enabled, ✗ = baseline; green bars = scorer runs)",
        fontsize=11,
    )
    ax.legend(fontsize=8, loc="upper right")
    plt.tight_layout()
    plt.savefig(ROOT / "eval" / "fig12_cross_run_scores.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Saved: eval/fig12_cross_run_scores.png")

    # ── A/B statistics (if both groups exist) ────────────────────────────────
    if "scorer_enabled" in final_df.columns and final_df["scorer_enabled"].nunique() == 2:
        print("\nA/B Comparison  (scorer_enabled vs baseline):")
        print(f"{'Metric':<26}  {'Scorer Avg':>12}  {'Baseline Avg':>13}  {'Delta':>8}")
        print("─" * 65)
        for col in avail_score_cols:
            s_avg = final_df[final_df["scorer_enabled"]][col].mean()
            b_avg = final_df[~final_df["scorer_enabled"]][col].mean()
            delta = s_avg - b_avg
            arrow = "↑" if delta > 0.01 else ("↓" if delta < -0.01 else "≈")
            print(f"  {SCORE_LABELS[col]:<24}  {s_avg:>12.4f}  {b_avg:>13.4f}  {arrow}{delta:>+7.4f}")
    else:
        print("\nℹ️  Need runs both with and without scorer for A/B comparison.")
        print("   Tip: temporarily comment out quality_scorer code in workflow.py and run once.")

    # ── OOC heatmap (if available) ────────────────────────────────────────────
    ooc_data = {}
    for _, row in final_df.iterrows():
        run_key = f"{str(row.get('game_title',''))[:12]}\n[{str(row.get('run_id',''))[-6:]}]"
        ooc_raw = row.get("ooc_scores")
        if isinstance(ooc_raw, dict) and ooc_raw:
            for char, score in ooc_raw.items():
                if char not in ooc_data:
                    ooc_data[char] = {}
                ooc_data[char][run_key] = score

    if ooc_data:
        chars    = list(ooc_data.keys())
        run_keys = list({k for v in ooc_data.values() for k in v.keys()})
        matrix   = np.array([
            [ooc_data[c].get(r, np.nan) for r in run_keys]
            for c in chars
        ])

        fig, ax = plt.subplots(figsize=(max(6, len(run_keys) * 2), max(3, len(chars) * 0.7 + 2)))
        sns.heatmap(
            matrix, ax=ax,
            xticklabels=run_keys, yticklabels=chars,
            cmap="RdYlGn", vmin=0, vmax=1,
            annot=True, fmt=".2f", linewidths=0.5,
            cbar_kws={"label": "OOC score (0=OOC, 1=in-character)"},
        )
        ax.axhline(y=len(chars), color="black", linewidth=0)
        # Highlight cells below threshold
        for i in range(len(chars)):
            for j in range(len(run_keys)):
                if not np.isnan(matrix[i, j]) and matrix[i, j] < 0.60:
                    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                               edgecolor="red", linewidth=2))
        ax.set_title(
            "OOC Score Heatmap  (red border = below 0.60 threshold)\n"
            "Rows = characters, Cols = generation runs",
            fontsize=11,
        )
        ax.tick_params(axis="x", rotation=0)
        plt.tight_layout()
        plt.savefig(ROOT / "eval" / "fig13_ooc_heatmap.png", dpi=130, bbox_inches="tight")
        plt.show()
        print("Saved: eval/fig13_ooc_heatmap.png")
    else:
        print("\nℹ️  No OOC data available. Run with OOC detection enabled (fan-fiction mode).")

    # ── Summary table ─────────────────────────────────────────────────────────
    print("\n" + "═" * 80)
    print("  Generation Quality Summary")
    print("═" * 80)
    display_cols = (
        ["run_id", "game_title", "scorer_enabled"]
        + [c for c in avail_score_cols if c in final_df.columns]
        + (["ooc_avg"] if "ooc_avg" in final_df.columns else [])
    )
    display_df = final_df[display_cols].copy()
    display_df.columns = [
        c.replace("_score", "").replace("_", " ").title()
        for c in display_df.columns
    ]
    print(display_df.to_string(index=False))

---
## Part 5 — Quality Scores & Benchmarks

Quantitative summary of the three core quality dimensions.
Each score is 0–1; dashed lines mark the minimum acceptable threshold.
Green = passes, Red = needs improvement.

In [ ]:
# ── Part 5: Quality Scores & Benchmarks ──────────────────────────────────────
# Import compute_metrics from auto_eval (no extra deps, pure Python)
import sys, os
sys.path.insert(0, str(ROOT))
from eval.auto_eval import compute_metrics, THRESHOLDS, SCORE_LABELS

metrics = compute_metrics(design, story_text)

# ── 5-A: Score bar chart ──────────────────────────────────────────────────────
score_keys = ["character_balance_score", "branch_balance_score", "scene_diversity_score"]
score_vals  = [metrics.get(k) for k in score_keys]
score_thresh = [THRESHOLDS[k] for k in score_keys]
labels       = [SCORE_LABELS[k] for k in score_keys]
colors       = ["#2ecc71" if (v is not None and v >= t) else "#e74c3c"
                for v, t in zip(score_vals, score_thresh)]

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(labels))
bars = ax.bar(x, [v if v is not None else 0 for v in score_vals],
              color=colors, width=0.45, zorder=3, edgecolor="white", linewidth=1.2)

for i, (t, v) in enumerate(zip(score_thresh, score_vals)):
    ax.hlines(t, i - 0.28, i + 0.28, colors="#555", linestyles="--", linewidth=1.5, zorder=4)
    ax.text(i + 0.30, t + 0.01, f"threshold={t}", va="bottom", fontsize=8.5, color="#555")
    if v is not None:
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")

import matplotlib.patches as mpatches
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0, 1.2)
ax.set_ylabel("Score  (0 = worst,  1 = best)", fontsize=11)
ax.set_title("Generated Game — Quality Score Overview", fontsize=13, fontweight="bold", pad=12)
ax.yaxis.grid(True, linestyle=":", alpha=0.5)
ax.set_axisbelow(True)
ax.legend(handles=[
    mpatches.Patch(color="#2ecc71", label="≥ threshold (pass)"),
    mpatches.Patch(color="#e74c3c", label="< threshold"),
], fontsize=9, loc="upper right")

plt.tight_layout()
plt.savefig(ROOT / "eval" / "fig_quality_scores.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eval/fig_quality_scores.png")

# ── 5-B: Structural feature table ─────────────────────────────────────────────
print("\nStructural features:")
for k, v in metrics["structural"].items():
    print(f"  {k:<24}: {v}")

print("\nText features:")
print(f"  lines_per_character   : {metrics['lines_per_character']}")
print(f"  expression_diversity  : {metrics['expression_diversity']}")
print(f"  scene_usage           : {metrics['scene_usage']}")

# ── 5-C: Radar chart (spider) ─────────────────────────────────────────────────
available_scores = {k: v for k, v in zip(score_keys, score_vals) if v is not None}
if len(available_scores) >= 3:
    import numpy as np
    ks   = list(available_scores.keys())
    vs   = list(available_scores.values())
    ts   = [THRESHOLDS[k] for k in ks]
    lbls = [SCORE_LABELS[k] for k in ks]

    N     = len(ks)
    theta = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    # close the polygon
    vs_c  = vs    + [vs[0]]
    ts_c  = ts    + [ts[0]]
    th_c  = theta + [theta[0]]

    fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw=dict(polar=True))
    ax.plot(th_c, vs_c, "o-", linewidth=2, color="#3498db", label="Score")
    ax.fill(th_c, vs_c, alpha=0.20, color="#3498db")
    ax.plot(th_c, ts_c, "--", linewidth=1.5, color="#e74c3c", label="Threshold")
    ax.fill(th_c, ts_c, alpha=0.08, color="#e74c3c")

    ax.set_xticks(theta)
    ax.set_xticklabels(lbls, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8)
    ax.set_title("Quality Radar Chart", fontsize=13, fontweight="bold", pad=18)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)

    plt.tight_layout()
    plt.savefig(ROOT / "eval" / "fig_quality_radar.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Saved: eval/fig_quality_radar.png")

---
## Part 6 — Producer Convergence Analysis

How do structural metrics evolve across Producer review rounds?

- Each point is one round of critique (Design → Producer → feedback → revision)
- Green dotted line = round where Producer said "PASS"
- Dashed grey line = recommended threshold
- **Ideally**: metrics improve monotonically and reach acceptable range before PASS

Data source: `logs/producer_rounds.jsonl` (written by workflow.py during generation)

In [ ]:
# ── Part 6: Producer Convergence ─────────────────────────────────────────────
import json
from pathlib import Path

ROUNDS_LOG = ROOT / "logs" / "producer_rounds.jsonl"

rounds = []
if ROUNDS_LOG.exists():
    with open(ROUNDS_LOG, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rounds.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

print(f"Producer rounds found in log: {len(rounds)}")

if not rounds:
    print("⚠️  No producer_rounds.jsonl found yet.")
    print("   This file is created automatically when you run `python main.py --mode create`.")
    print("   Re-run this cell after generating a game.")
else:
    round_nums = [r["round"] for r in rounds]
    verdicts   = [r.get("verdict", "") for r in rounds]
    pass_idx   = [i for i, v in enumerate(verdicts) if v == "PASS"]

    # ── 6-A: 4-panel metric progression ──────────────────────────────────────
    panel_metrics = [
        ("node_count",       "Node Count",           None),
        ("branching_factor", "Branching Factor",     None),
        ("path_entropy",     "Path Entropy\n(branch balance)", THRESHOLDS["branch_balance_score"]),
        ("merge_ratio",      "Merge Ratio\n(DAG convergence)", None),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    axes_flat = axes.flatten()

    for ax, (key, label, thresh) in zip(axes_flat, panel_metrics):
        vals = [r.get(key) for r in rounds]
        valid = [(rn, v) for rn, v in zip(round_nums, vals) if v is not None]
        if not valid:
            ax.set_visible(False)
            continue
        xs, ys = zip(*valid)
        ax.plot(xs, ys, marker="o", linewidth=2.2, markersize=9,
                color=PALETTE[panel_metrics.index((key, label, thresh)) % len(PALETTE)])

        for xi, yi in zip(xs, ys):
            ax.annotate(f"{yi:.3f}" if isinstance(yi, float) else str(yi),
                        (xi, yi), textcoords="offset points",
                        xytext=(0, 9), ha="center", fontsize=9)

        if thresh is not None:
            ax.axhline(thresh, color="grey", linestyle="--", linewidth=1.3,
                       label=f"threshold={thresh}")
            ax.legend(fontsize=8)

        for pi in pass_idx:
            if pi < len(xs):
                ax.axvline(xs[pi], color="#27ae60", linestyle=":", linewidth=2, alpha=0.8)
                y_top = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else (max(ys) * 1.05)
                ax.text(xs[pi] - 0.08, y_top * 0.97, "PASS",
                        rotation=90, va="top", ha="right",
                        color="#27ae60", fontsize=8, fontweight="bold")

        ax.set_ylabel(label, fontsize=10)
        ax.set_xticks(list(xs))
        ax.yaxis.grid(True, linestyle=":", alpha=0.45)
        ax.set_axisbelow(True)

    for ax in axes_flat:
        ax.set_xlabel("Producer Round", fontsize=10)

    fig.suptitle("Producer Review Loop — Metric Progression per Round",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(ROOT / "eval" / "fig_producer_rounds.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Saved: eval/fig_producer_rounds.png")

    # ── 6-B: Verdict timeline ─────────────────────────────────────────────────
    print("\nProducer verdict timeline:")
    for r in rounds:
        verdict_short = r["verdict"] if r["verdict"] == "PASS" else r["verdict"][:80] + "..."
        print(f"  Round {r['round']:>2}  →  {verdict_short}")

    print(f"\nTotal rounds: {len(rounds)}  |  PASS at round: {pass_idx[-1]+1 if pass_idx else 'N/A (max reached)'}")

    # ── 6-C: Convergence assessment ───────────────────────────────────────────
    if len(rounds) >= 2:
        first_pe  = rounds[0].get("path_entropy", None)
        last_pe   = rounds[-1].get("path_entropy", None)
        first_bf  = rounds[0].get("branching_factor", None)
        last_bf   = rounds[-1].get("branching_factor", None)

        print("\nConvergence summary (first → last round):")
        if first_pe is not None and last_pe is not None:
            delta = last_pe - first_pe
            icon  = "↑" if delta > 0 else ("↓" if delta < 0 else "→")
            print(f"  Path Entropy   : {first_pe:.4f} → {last_pe:.4f}  {icon} {delta:+.4f}")
        if first_bf is not None and last_bf is not None:
            delta = last_bf - first_bf
            icon  = "↑" if delta > 0 else ("↓" if delta < 0 else "→")
            print(f"  Branching Factor: {first_bf:.3f} → {last_bf:.3f}  {icon} {delta:+.3f}")